In [1]:
"""
process_tc_averages.py
Run in Jupyter: %run process_tc_averages.py
Or from terminal: python process_tc_averages.py
"""

import os
import re
import glob
import pandas as pd
from io import StringIO
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── SET YOUR DIRECTORY HERE ────────────────────────────────────────────────────
BASE_DIR = r"D:\NRC\FCAI_combined\effusion_coupon\experiment\thermocouple"
# ──────────────────────────────────────────────────────────────────────────────

TC_COLS     = [f"TC{i}" for i in range(17, 29)]
OUTPUT_NAME = "time_average_summary.xlsx"


def parse_metadata(filename):
    stem = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r"fcai_eff1_(\d+)_(\d+)_(\d+)", stem, re.IGNORECASE)
    if not m:
        return {"filename": stem, "equiv_ratio": None, "flow_slpm": None, "run_no": None}
    er_raw, flow, run_no = m.group(1), m.group(2), m.group(3)
    er_int = int(er_raw)
    equiv_ratio = er_int / 10.0 if er_int < 20 else er_int / 100.0
    return {"filename": stem, "equiv_ratio": equiv_ratio,
            "flow_slpm": int(flow), "run_no": int(run_no)}


def read_tc_file(filepath):
    """
    Read tab-delimited file, trimming each data row to exactly the number of
    columns defined in the header. This handles files where data rows have
    extra trailing fields that would shift column alignment in pandas.
    """
    with open(filepath, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if not lines:
        return pd.DataFrame()

    header   = lines[0].rstrip("\n")
    n_cols   = len(header.split("\t"))
    trimmed  = [header]

    for line in lines[1:]:
        fields = line.rstrip("\n").split("\t")
        # Keep only the first n_cols fields; pad with empty string if shorter
        fields = (fields[:n_cols] + [""] * n_cols)[:n_cols]
        trimmed.append("\t".join(fields))

    df = pd.read_csv(StringIO("\n".join(trimmed)), sep="\t", header=0, engine="python")
    df.columns = [c.strip() for c in df.columns]

    present = [c for c in TC_COLS if c in df.columns]
    if not present:
        return pd.DataFrame()

    return df[present].apply(pd.to_numeric, errors="coerce")


def compute_averages(df):
    return {tc: float(df[tc].mean()) if tc in df.columns else float("nan")
            for tc in TC_COLS}


# ── Gather files ───────────────────────────────────────────────────────────────
files = sorted(
    glob.glob(os.path.join(BASE_DIR, "fcai_eff1_*.xls")) +
    glob.glob(os.path.join(BASE_DIR, "fcai_eff1_*.csv"))
)
print(f"Found {len(files)} file(s)")

rows = []
for fp in files:
    meta = parse_metadata(fp)
    df   = read_tc_file(fp)
    if df.empty:
        print(f"  [SKIP] {meta['filename']} — no TC columns found")
        continue
    avgs = compute_averages(df)
    rows.append({**meta, **avgs})
    print(f"  [OK]   {meta['filename']}  ER={meta['equiv_ratio']}  "
          f"flow={meta['flow_slpm']} slpm  n={len(df)}  "
          f"TC17={avgs['TC17']:.3f}  TC28={avgs['TC28']:.3f}")

if not rows:
    print("No data rows collected — check file format.")
else:
    summary = pd.DataFrame(rows).sort_values(
        ["equiv_ratio", "flow_slpm", "run_no"], ignore_index=True)

    # ── Write Excel ────────────────────────────────────────────────────────────
    wb = Workbook()
    ws = wb.active
    ws.title = "TC Time Averages"

    hdr_font  = Font(name="Arial", bold=True, color="FFFFFF", size=11)
    hdr_fill  = PatternFill("solid", start_color="2F5496")
    meta_fill = PatternFill("solid", start_color="D9E1F2")
    alt_fill  = PatternFill("solid", start_color="EEF3FB")
    data_font = Font(name="Arial", size=10)
    center    = Alignment(horizontal="center", vertical="center", wrap_text=True)
    thin      = Side(style="thin", color="B8CCE4")
    border    = Border(left=thin, right=thin, top=thin, bottom=thin)

    headers = ["Case", "Equiv. Ratio (φ)", "Flow Rate (slpm)", "Run No."] + TC_COLS
    for col_idx, h in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=h)
        cell.font = hdr_font; cell.fill = hdr_fill
        cell.alignment = center; cell.border = border

    for row_idx, row in summary.iterrows():
        r = row_idx + 2
        for col_idx, val in enumerate(
                [row["filename"], row["equiv_ratio"], row["flow_slpm"], row["run_no"]], 1):
            cell = ws.cell(row=r, column=col_idx, value=val)
            cell.font = data_font; cell.fill = meta_fill
            cell.alignment = center; cell.border = border

        for col_idx, tc in enumerate(TC_COLS, 5):
            val = row.get(tc)
            cell = ws.cell(row=r, column=col_idx,
                           value=round(val, 3) if pd.notna(val) else "N/A")
            cell.font = data_font; cell.alignment = center; cell.border = border
            if row_idx % 2 == 0:
                cell.fill = alt_fill

    ws.column_dimensions["A"].width = 30
    ws.column_dimensions["B"].width = 16
    ws.column_dimensions["C"].width = 18
    ws.column_dimensions["D"].width = 10
    for i in range(5, 5 + len(TC_COLS)):
        ws.column_dimensions[get_column_letter(i)].width = 12
    ws.freeze_panes = "A2"

    output_path = os.path.join(BASE_DIR, OUTPUT_NAME)
    wb.save(output_path)
    print(f"\nDone! Summary saved to:\n  {output_path}")

Found 24 file(s)
  [OK]   fcai_eff1_085_375_186  ER=0.85  flow=375 slpm  n=120  TC17=362.319  TC28=388.912
  [OK]   fcai_eff1_085_500_187  ER=0.85  flow=500 slpm  n=120  TC17=346.474  TC28=373.626
  [OK]   fcai_eff1_085_625_188  ER=0.85  flow=625 slpm  n=120  TC17=322.981  TC28=348.976
  [OK]   fcai_eff1_09_1125_193  ER=0.9  flow=1125 slpm  n=120  TC17=310.153  TC28=398.690
  [OK]   fcai_eff1_09_1380_194  ER=0.9  flow=1380 slpm  n=120  TC17=263.496  TC28=348.367
  [OK]   fcai_eff1_09_375_189  ER=0.9  flow=375 slpm  n=120  TC17=419.247  TC28=482.862
  [OK]   fcai_eff1_09_500_190  ER=0.9  flow=500 slpm  n=120  TC17=408.161  TC28=487.720
  [OK]   fcai_eff1_09_625_191  ER=0.9  flow=625 slpm  n=120  TC17=394.378  TC28=477.363
  [OK]   fcai_eff1_09_860_192  ER=0.9  flow=860 slpm  n=120  TC17=353.781  TC28=436.828
  [OK]   fcai_eff1_10_1125_196  ER=1.0  flow=1125 slpm  n=120  TC17=361.322  TC28=492.100
  [OK]   fcai_eff1_10_1380_195  ER=1.0  flow=1380 slpm  n=120  TC17=347.692  TC28=482.002
 

In [2]:
# import glob, os

# base_dir = r"D:\NRC\FCAI_combined\effusion_coupon\experiment\thermocouple"
# files = glob.glob(os.path.join(base_dir, "fcai_eff1_*.xls")) + \
#         glob.glob(os.path.join(base_dir, "fcai_eff1_*.csv"))
# print(f"Found {len(files)} files")
# print(files[:5])